# Fermeture $w_{sec}\partial_z\bar u$ du flux de Reynolds — correction $\theta/T$, extension verticale, cadre bulk-plume (Romps 2014)
## Méso-NH RCE `large300`

Notebook indépendant. Reprend le setup commun et refait, dans l'ordre, tous les calculs du notebook précédent (sans les réafficher un par un), pour arriver directement à la comparaison finale 2–8 km. La suite répond point par point aux questions posées sur papier : correction $\theta/T$ de l'équilibre radiatif-subsident, extension au-dessus de 8 km et abandon en dessous de 1.5–2 km, comparaison $\theta_c$ (nuage) vs $\bar\theta$ (environnement) via un diagnostic d'entraînement à la Romps (2014), test des hypothèses H1/H2 sur le bilan de moment en flux de masse, et une métrique de comparaison dépendante de $z$ plus informative qu'une simple corrélation globale.

## §0. Setup commun (identique aux notebooks précédents)

In [ ]:
# ============================================================
#  SETUP COMMUN
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.ndimage import uniform_filter1d

plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.grid':True,
                     'grid.alpha':0.25,'image.cmap':'RdBu_r'})
Rd,Rv=287.05,461.5; EPSILON=Rd/Rv; g=9.81; cp=1004.0; p_ref=1e5
Z_TOP_KM=15.0
DIR_3D,DIR_2D,DIR_1D='3D','2D','1D'
def path3d(v): return os.path.join(DIR_3D,f'MESONH_RCE_large300_3D_{v}.nc')
def path2d(v): return os.path.join(DIR_2D,f'MESONH_RCE_large300_2D_{v}.nc')
def path1d(v): return os.path.join(DIR_1D,f'MESONH_RCE_large300_1D_{v}.nc')
BLOC=2

_ds=xr.open_dataset(path3d('ua')); _da=_ds['ua']
dim_t,dim_z,dim_y,dim_x=_da.dims[:4]
n_t,n_z=_da.sizes[dim_t],_da.sizes[dim_z]; n_y,n_x=_da.sizes[dim_y],_da.sizes[dim_x]
_ds.close(); del _ds,_da; gc.collect()
_ds1=xr.open_dataset(path1d('ua_avg')); alt=_ds1.altitude.values.astype(float).copy(); _ds1.close()
zkm=alt/1000.0
mask_show=alt<=Z_TOP_KM*1000; m28=(alt>=2000)&(alt<=8000); m812=(alt>=8000)&(alt<=12000)
m015=(alt>=0)&(alt<=2000)

t_stat=0; idx_stat=slice(t_stat,None); n_stat=n_t-t_stat
dt_phys=6*3600.0
tt=np.arange(n_stat)

ds_prw=xr.open_dataset(path2d('prw')); prw_all=ds_prw['prw'].load(); ds_prw.close()
prw_mean=prw_all.isel({dim_t:idx_stat}).mean(dim=dim_t)
PRW_SEUIL=float(np.median(prw_all.values.ravel()))
mh=(prw_mean.values>PRW_SEUIL); ms=~mh; mh_flat=mh.ravel(); ms_flat=ms.ravel()
f_h=float(mh.mean()); f_s=float(ms.mean())
del prw_all; gc.collect()

def smooth_t(fld, win=8):
    return uniform_filter1d(fld, size=win, axis=1, mode='nearest')

print(f'{n_t}t x {n_z}z x {n_y}y x {n_x}x | humide={f_h:.0%} sec={f_s:.0%}')


## §1. Recalcul groupé des champs du notebook précédent (silencieux)

In [ ]:
# ============================================================
#  rho0(z), theta(z,t), T(z,t)  — domaine total
# ============================================================
rho0_sum=np.zeros(n_z); n_rho=0
theta_zt=np.zeros((n_z,n_stat)); T_zt=np.zeros((n_z,n_stat))
ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa')); ds_hus=xr.open_dataset(path3d('hus'))
for t0 in range(t_stat,n_t,BLOC):
    t1=min(t0+BLOC,n_t); sl={dim_t:slice(t0,t1)}
    T=ds_ta['ta'].isel(sl).values; p=ds_pa['pa'].isel(sl).values; qv=ds_hus['hus'].isel(sl).values
    Tv=T*(1+qv/EPSILON)/(1+qv); rho=p/(Rd*Tv)
    rho0_sum+=rho.mean(axis=(0,2,3))*(t1-t0); n_rho+=(t1-t0)
    del T,p,qv,Tv,rho; gc.collect()
ds_ta.close(); ds_pa.close(); ds_hus.close(); rho0=rho0_sum/n_rho

ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa'))
for iz in range(n_z):
    T=ds_ta['ta'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    p=ds_pa['pa'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    theta=T*(p_ref/p)**(Rd/cp)
    theta_zt[iz]=theta.mean(axis=1); T_zt[iz]=T.mean(axis=1)
    del T,p,theta
ds_ta.close(); ds_pa.close(); gc.collect()
dthetadz_zt=np.gradient(theta_zt, alt, axis=0)
print('rho0, theta_zt, T_zt, dthetadz_zt : ok')


In [ ]:
# ============================================================
#  ubar_zt, wbar_zt, flux_zt (u'w') — domaine total
#  + u_h_zt/u_s_zt, w_h_zt/w_s_zt — decomposition humide/sec
# ============================================================
ubar_zt=np.zeros((n_z,n_stat)); wbar_zt=np.zeros((n_z,n_stat)); flux_zt=np.zeros((n_z,n_stat))
u_h_zt=np.zeros((n_z,n_stat)); u_s_zt=np.zeros((n_z,n_stat))
w_h_zt=np.zeros((n_z,n_stat)); w_s_zt=np.zeros((n_z,n_stat))
ds_u=xr.open_dataset(path3d('ua')); ds_w=xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    u=ds_u['ua'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    w=ds_w['wa'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    ub=u.mean(axis=1); wb=w.mean(axis=1); up=u-ub[:,None]; wp=w-wb[:,None]
    ubar_zt[iz]=ub; wbar_zt[iz]=wb; flux_zt[iz]=rho0[iz]*(up*wp).mean(axis=1)
    u_h_zt[iz]=u[:,mh_flat].mean(axis=1); u_s_zt[iz]=u[:,ms_flat].mean(axis=1)
    w_h_zt[iz]=w[:,mh_flat].mean(axis=1); w_s_zt[iz]=w[:,ms_flat].mean(axis=1)
    del u,w,ub,wb,up,wp; gc.collect()
ds_u.close(); ds_w.close(); gc.collect()

dudz_tot=np.gradient(ubar_zt, alt, axis=0)
uw_zt=flux_zt/rho0[:,None]
terme_flux=smooth_t(-(1/rho0[:,None])*np.gradient(flux_zt, alt, axis=0))
dudz_tot_s=smooth_t(dudz_tot); w_s_obs_s=smooth_t(w_s_zt)
print('ubar_zt, wbar_zt, flux_zt, u_h/s_zt, w_h/s_zt, terme_flux : ok')


In [ ]:
# ============================================================
#  tntr(z,t) domaine total — tendance radiative
# ============================================================
tntr_zt=np.zeros((n_z,n_stat))
ds_tntr=xr.open_dataset(path3d('tntr'))
for iz in range(n_z):
    tr=ds_tntr['tntr'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    tntr_zt[iz]=tr.mean(axis=1)
    del tr
ds_tntr.close(); gc.collect()

with np.errstate(divide='ignore', invalid='ignore'):
    w_sec_rce_brut     = -tntr_zt/dthetadz_zt                      # sans correction theta/T
    w_sec_rce_corrige  = -(theta_zt/T_zt)*tntr_zt/dthetadz_zt      # avec correction theta/T (H0)
w_sec_rce_brut_s    = smooth_t(w_sec_rce_brut)
w_sec_rce_corrige_s = smooth_t(w_sec_rce_corrige)
w_sec_rce_x_shear    = w_sec_rce_corrige_s*dudz_tot_s
print('tntr_zt, w_sec_rce (brut+corrige), fermeture finale : ok')


## §2. Affichage final — fermeture complète sur 2–8 km

In [ ]:
# ============================================================
#  terme_flux vs w_sec_rce_corrige * dudz_tot, 2-8km
# ============================================================
fig, axes = plt.subplots(1,2,figsize=(11,5),sharey=True)
v = max(np.percentile(np.abs(terme_flux[m28]),98), np.percentile(np.abs(w_sec_rce_x_shear[m28]),98))*3600+1e-12
for ax,(fld,lab) in zip(axes,[(w_sec_rce_x_shear*3600, r"$-(\theta/T)\dot T_{rad}/\partial_z\bar\theta\ \partial_z\bar u$ (m/s/h)"),
                                (terme_flux*3600, r"$-\frac{1}{\rho_0}\partial_z(\rho_0\overline{u'w'})$ (m/s/h)")]):
    im=ax.pcolormesh(tt+t_stat, zkm[m28], fld[m28], cmap='RdBu_r',
                      norm=TwoSlopeNorm(0,-v,v), shading='auto')
    ax.set_xlabel('t'); ax.set_title(lab,fontsize=10)
    fig.colorbar(im,ax=ax,pad=.02)
axes[0].set_ylabel('z (km)')
plt.suptitle('Fermeture radiative-subsidente (theta/T corrigee) vs flux reel (2-8km)', fontweight='bold')
plt.tight_layout(); plt.show()

R2_final=np.full(n_z,np.nan)
for iz in range(n_z):
    if not (2000<=alt[iz]<=8000): continue
    a,b=terme_flux[iz,:], w_sec_rce_x_shear[iz,:]
    if np.nanstd(a)<1e-15: continue
    ss_res=np.nansum((a-b)**2); ss_tot=np.nansum((a-a.mean())**2)
    R2_final[iz]=1-ss_res/ss_tot if ss_tot>0 else np.nan
print(f'R2 median (2-8km), fermeture finale corrigee : {np.nanmedian(R2_final[m28]):.2f}')


## §3. La correction $\theta/T$ améliore-t-elle vraiment l'ajustement ? (H0 du tableau)

$$w_{sec}\,\partial_z\bar\theta = \frac{\theta}{T}\dot T_{rad} \quad\text{(H0, tableau)}$$

On compare $w_{sec}$ observé (zone sèche) à sa version prédite **sans** le facteur $\theta/T$ (ce que le premier essai avait fait par erreur) et **avec** ce facteur — le facteur croit avec l'altitude, donc l'écart entre les deux versions doit se creuser en montant.

In [ ]:
# ============================================================
#  w_sec observe vs predit, AVEC et SANS correction theta/T
# ============================================================
R2_brut=np.full(n_z,np.nan); R2_corrige=np.full(n_z,np.nan)
for iz in range(n_z):
    if not (2000<=alt[iz]<=8000): continue
    a=w_s_obs_s[iz,:]
    for b,store in [(w_sec_rce_brut_s[iz,:],'brut'),(w_sec_rce_corrige_s[iz,:],'corrige')]:
        if np.nanstd(a)<1e-15: continue
        ss_res=np.nansum((a-b)**2); ss_tot=np.nansum((a-a.mean())**2)
        r2=1-ss_res/ss_tot if ss_tot>0 else np.nan
        if store=='brut': R2_brut[iz]=r2
        else: R2_corrige[iz]=r2

fig,ax=plt.subplots(figsize=(6,6))
ax.plot(R2_brut[m28], zkm[m28], 'o-', ms=3, label='sans correction theta/T')
ax.plot(R2_corrige[m28], zkm[m28], 'o-', ms=3, label='avec correction theta/T')
ax.axvline(0,color='k',lw=.5); ax.legend(fontsize=9)
ax.set_xlabel('R2 (w_sec observe vs predit)'); ax.set_ylabel('z (km)')
ax.set_title('Effet de la correction theta/T (H0)')
plt.tight_layout(); plt.show()
print(f'R2 median SANS correction : {np.nanmedian(R2_brut[m28]):.2f}')
print(f'R2 median AVEC correction : {np.nanmedian(R2_corrige[m28]):.2f}')


## §4. Extension verticale — 8–12 km, et abandon en dessous de 2 km

Même comparaison, hors de la bande 2–8 km : au-dessus (8–12 km, proche du sommet convectif, mélange désordonné attendu) et en dessous (0–2 km, couche de surface, turbulence mécanique attendue plutôt que subsidence radiative).

In [ ]:
# ============================================================
#  R2 par bande : 0-2km / 2-8km / 8-12km
# ============================================================
def r2_bande(mask):
    r2=np.full(n_z,np.nan)
    for iz in range(n_z):
        if not mask[iz]: continue
        a,b=terme_flux[iz,:], w_sec_rce_x_shear[iz,:]
        if np.nanstd(a)<1e-15: continue
        ss_res=np.nansum((a-b)**2); ss_tot=np.nansum((a-a.mean())**2)
        r2[iz]=1-ss_res/ss_tot if ss_tot>0 else np.nan
    return r2

R2_015=r2_bande(m015); R2_28=r2_bande(m28); R2_812=r2_bande(m812)

fig,ax=plt.subplots(figsize=(6,7))
ax.plot(R2_015[m015|m28|m812], zkm[m015|m28|m812], 'o-', ms=3, color='0.6')
ax.plot(R2_015[m015], zkm[m015], 'o-', ms=4, color='C0', label='0-2km')
ax.plot(R2_28[m28], zkm[m28], 'o-', ms=4, color='C1', label='2-8km')
ax.plot(R2_812[m812], zkm[m812], 'o-', ms=4, color='C2', label='8-12km')
ax.axvline(0,color='k',lw=.5); ax.legend(fontsize=9)
ax.set_xlabel('R2'); ax.set_ylabel('z (km)')
ax.set_title('Ou la fermeture radiative-subsidente tient-elle ?')
plt.tight_layout(); plt.show()

print(f'R2 median 0-2km  : {np.nanmedian(R2_015[m015]):.2f}')
print(f'R2 median 2-8km  : {np.nanmedian(R2_28[m28]):.2f}')
print(f'R2 median 8-12km : {np.nanmedian(R2_812[m812]):.2f}')


## §5. $\theta_c$ (nuage) vs $\bar\theta$ (environnement) — diagnostic d'entraînement à la Romps (2014)

Masque de nuage explicite (pas le masque PRW) : colonne en ascendance convective si $w>1\,$m/s **et** contenu en eau condensée (liquide+glace) $>10^{-5}\,$kg/kg — critère de Romps & Kuang (2010), repris dans Romps (2014). $\theta_c(z,t)$ est la moyenne de $\theta$ sur ces colonnes. Le taux d'entraînement composite $\varepsilon(z)$ est diagnostiqué par l'équation (17) de Romps (2014), adaptée à $\theta$ : $\varepsilon=\partial_z\theta_c/(\bar\theta-\theta_c)$.

In [ ]:
# ============================================================
#  Masque de nuage explicite (w>1m/s, condensat>1e-5) + theta_c(z,t), u_c(z,t)
# ============================================================
SEUIL_W=1.0; SEUIL_COND=1e-5
theta_c_zt=np.full((n_z,n_stat), np.nan); u_c_zt=np.full((n_z,n_stat), np.nan)
frac_cloud_zt=np.zeros((n_z,n_stat))
ds_w=xr.open_dataset(path3d('wa')); ds_u=xr.open_dataset(path3d('ua'))
ds_clw=xr.open_dataset(path3d('clw')); ds_cli=xr.open_dataset(path3d('cli'))
ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa'))
for iz in range(n_z):
    w=ds_w['wa'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    u=ds_u['ua'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    clw=ds_clw['clw'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    cli=ds_cli['cli'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    T=ds_ta['ta'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    p=ds_pa['pa'].isel({dim_t:idx_stat,dim_z:iz}).values.reshape(n_stat,-1)
    theta=T*(p_ref/p)**(Rd/cp)
    cloud=(w>SEUIL_W)&((clw+cli)>SEUIL_COND)
    for it in range(n_stat):
        m=cloud[it,:]; frac_cloud_zt[iz,it]=m.mean()
        if m.sum()>10:
            theta_c_zt[iz,it]=theta[it,m].mean(); u_c_zt[iz,it]=u[it,m].mean()
    del w,u,clw,cli,T,p,theta,cloud; gc.collect()
ds_w.close(); ds_u.close(); ds_clw.close(); ds_cli.close(); ds_ta.close(); ds_pa.close(); gc.collect()

fig,ax=plt.subplots(figsize=(6,6))
ax.plot(np.nanmean(frac_cloud_zt,axis=1)[mask_show]*100, zkm[mask_show])
ax.set_xlabel('fraction nuageuse moyenne (%)'); ax.set_ylabel('z (km)'); ax.set_ylim(0,Z_TOP_KM)
ax.set_title('Fraction de colonnes en ascendance nuageuse (w>1m/s, condensat>1e-5)')
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Entrainement composite epsilon(z) : dz(theta_c) = eps*(theta_bar - theta_c)
# ============================================================
dtheta_c_dz = np.gradient(theta_c_zt, alt, axis=0)
denom = theta_zt - theta_c_zt
SEUIL_DENOM = 0.3*np.nanpercentile(np.abs(denom[m28]), 90)
with np.errstate(divide='ignore', invalid='ignore'):
    eps_zt = np.where(np.abs(denom)>SEUIL_DENOM, dtheta_c_dz/denom, np.nan)
eps_z = np.nanmedian(eps_zt, axis=1)   # profil composite, comme Romps Fig.6

fig,ax=plt.subplots(figsize=(6,7))
ax.plot(eps_z[mask_show]*1000, zkm[mask_show], 'o-', ms=3)
ax.set_xlabel(r'$\varepsilon(z)$ ($10^{-3}$ m$^{-1}$)'); ax.set_ylabel('z (km)')
ax.set_title("Taux d'entrainement composite (Romps eq.17, applique a theta)")
plt.tight_layout(); plt.show()

# comparaison theta_c vs theta_bar dans la fermeture w_sec : remplacer theta_bar par theta_c
dtheta_c_dz_s = smooth_t(dtheta_c_dz)
with np.errstate(divide='ignore', invalid='ignore'):
    w_sec_rce_thetac = -(theta_c_zt/T_zt)*tntr_zt/dtheta_c_dz
w_sec_rce_thetac_s = smooth_t(w_sec_rce_thetac)

R2_thetac=np.full(n_z,np.nan)
for iz in range(n_z):
    if not (2000<=alt[iz]<=8000): continue
    a,b=w_s_obs_s[iz,:], w_sec_rce_thetac_s[iz,:]
    if np.nanstd(a)<1e-15 or np.all(np.isnan(b)): continue
    ss_res=np.nansum((a-b)**2); ss_tot=np.nansum((a-a.mean())**2)
    R2_thetac[iz]=1-ss_res/ss_tot if ss_tot>0 else np.nan

fig,ax=plt.subplots(figsize=(6,6))
ax.plot(R2_corrige[m28], zkm[m28], 'o-', ms=3, label='theta_bar (environnement)')
ax.plot(R2_thetac[m28], zkm[m28], 'o-', ms=3, label='theta_c (nuage)')
ax.axvline(0,color='k',lw=.5); ax.legend(fontsize=9)
ax.set_xlabel('R2 (w_sec observe vs predit)'); ax.set_ylabel('z (km)')
ax.set_title('theta_c ameliore-t-il la fermeture par rapport a theta_bar ?')
plt.tight_layout(); plt.show()


## §6. Bilan en flux de masse (tableau blanc) — test des hypothèses H1 et H2

Dérivation du tableau (bilan sur une tranche $\delta z$, à la Romps 2012b/2014) :

$$T_u = -w_{sec}\,\partial_z\bar u + D\,(u_c-\bar u)$$

**(H1)** $|D(u_c-\bar u)| \ll |w_{sec}\partial_z\bar u| \Rightarrow T_u\approx w_{sec}\partial_z\bar u$ (c'est l'approximation qu'on a testée implicitement jusqu'ici).

**(H2)** $D=\partial_z w_{sec}$ (continuité de masse) et $\varepsilon\ll D$.

In [ ]:
# ============================================================
#  H2 : D = dz(w_sec), comparaison a epsilon
# ============================================================
D_zt = np.gradient(w_s_obs_s, alt, axis=0)   # (H2)
D_z  = np.nanmedian(D_zt, axis=1)

fig,ax=plt.subplots(figsize=(6,7))
ax.plot(D_z[mask_show]*1000, zkm[mask_show], 'o-', ms=3, label='D = dz(w_sec)  (H2)')
ax.plot(eps_z[mask_show]*1000, zkm[mask_show], 'o-', ms=3, label='epsilon (Romps eq.17)')
ax.axvline(0,color='k',lw=.5); ax.legend(fontsize=9)
ax.set_xlabel(r'($10^{-3}$ m$^{-1}$)'); ax.set_ylabel('z (km)')
ax.set_title('Test H2 : epsilon << D ?')
plt.tight_layout(); plt.show()

ratio_eps_D = np.abs(eps_z)/(np.abs(D_z)+1e-15)
print('ratio median |eps|/|D| (2-8km) :', np.nanmedian(ratio_eps_D[m28]),
      '(<<1 si H2 verifiee)')


In [ ]:
# ============================================================
#  H1 : |D(uc-ubar)| vs |w_sec dz(ubar)|
# ============================================================
terme_detrainement = D_zt * (u_c_zt - ubar_zt)
terme_advectif      = w_s_obs_s * dudz_tot_s

fig, axes = plt.subplots(1,2,figsize=(11,5),sharey=True)
v = max(np.nanpercentile(np.abs(terme_detrainement[m28]),95),
        np.nanpercentile(np.abs(terme_advectif[m28]),95))*3600+1e-12
for ax,(fld,lab) in zip(axes,[(terme_advectif*3600, r"$w_{sec}\partial_z\bar u$ (m/s/h)"),
                                (terme_detrainement*3600, r"$D(u_c-\bar u)$ (m/s/h)")]):
    im=ax.pcolormesh(tt+t_stat, zkm[m28], fld[m28], cmap='RdBu_r',
                      norm=TwoSlopeNorm(0,-v,v), shading='auto')
    ax.set_xlabel('t'); ax.set_title(lab,fontsize=10)
    fig.colorbar(im,ax=ax,pad=.02)
axes[0].set_ylabel('z (km)')
plt.suptitle('Test H1 : le terme de detrainement est-il negligeable ?', fontweight='bold')
plt.tight_layout(); plt.show()

rms_adv = np.sqrt(np.nanmean(terme_advectif[m28]**2))
rms_det = np.sqrt(np.nanmean(terme_detrainement[m28]**2))
print(f'RMS w_sec*dudz (2-8km)   : {rms_adv*3600:.4f} m/s/h')
print(f'RMS D*(uc-ubar) (2-8km)  : {rms_det*3600:.4f} m/s/h')
print(f'ratio detrainement/advectif : {rms_det/rms_adv:.2f} (<<1 si H1 verifiee)')


## §7. Une métrique de comparaison dépendante de $z$, plus informative qu'une corrélation simple

Score normalisé glissant en temps : NRMSE local $=\dfrac{\text{RMSE}(z,t)}{\sigma_{\text{observe}}(z)}$, calculé sur une fenêtre glissante — donne un profil $(z,t)$ de qualité d'ajustement plutôt qu'un chiffre unique par niveau, et permet de voir *quand* la fermeture décroche, pas seulement *si*.

In [ ]:
# ============================================================
#  NRMSE glissant (z,t) entre terme_flux et w_sec_rce_x_shear
# ============================================================
WIN_SCORE=10
nrmse_zt = np.full((n_z,n_stat), np.nan)
sigma_z = np.nanstd(terme_flux[:,:], axis=1)
for iz in range(n_z):
    if sigma_z[iz]<1e-15: continue
    err2 = (terme_flux[iz,:]-w_sec_rce_x_shear[iz,:])**2
    err2_roll = uniform_filter1d(err2, size=WIN_SCORE, mode='nearest')
    nrmse_zt[iz] = np.sqrt(err2_roll)/sigma_z[iz]

fig,ax=plt.subplots(figsize=(8,5)); ax.grid(False)
im=ax.pcolormesh(tt+t_stat, zkm[m28], nrmse_zt[m28], cmap='viridis_r', vmin=0, vmax=2, shading='auto')
ax.set_xlabel('t'); ax.set_ylabel('z (km)')
ax.set_title('NRMSE glissant (z,t) — 0=parfait, 1=erreur=variabilite du signal')
fig.colorbar(im,ax=ax,pad=.02)
plt.tight_layout(); plt.show()

print(f'NRMSE median (2-8km) : {np.nanmedian(nrmse_zt[m28]):.2f}')
